# 01 · Pipeline: del operacional a RAW, y de RAW a MASTER

**Caso transversal:** Online Retail II. *Dado un cliente y una fecha de corte, ¿volverá a comprar en los 90 días siguientes?*

**Objetivo de este cuaderno:** traer los datos del sistema de ventas (`retail.db`) a la base analítica (`curso.db`) y convertirlos en un dato
**fiable**: sin duplicados, con tipos coherentes, con cada línea marcada como venta válida o no, y con reglas de calidad que lo demuestran.

Lo hace en dos capas, cada una con una responsabilidad distinta:

| Capa | Qué es | Su regla | Tablas |
|---|---|---|---|
| **RAW** (dato crudo) | La **copia fiel** del sistema de origen, tal como llegó | **No se corrige nada.** Si el origen tiene un error, RAW lo conserva | `raw_clientes`, `raw_productos`, `raw_facturas`, `raw_lineas_factura`, `raw_carga` |
| **MASTER** (dato maestro) | El dato **depurado y relacional**, el que todos pueden usar | Se puede borrar y **regenerar desde RAW**. Lo que se descarta deja rastro | `master_clientes`, `master_productos`, `master_facturas`, `master_lineas`, `master_descartes`, `master_calidad` |

**¿Por qué dos capas y no limpiar directamente al copiar?** Porque las reglas de limpieza cambian (hoy decides que un precio 0 no es una venta;
mañana, que sí). Si RAW guarda el original intacto, puedes **rehacer MASTER** con las reglas nuevas sin volver a pedir los datos al sistema de origen,
y siempre puedes demostrar de dónde salió cada cifra. En la literatura se las llama a menudo capas *bronze* (RAW) y *silver* (MASTER).

**Entra:** `data/external/retail.db` (lo crea el cuaderno 00). **Sale:** las capas RAW y MASTER en `data/processed/curso.db`.

Antes de empezar, en VS Code elige el kernel **env-ia-mlops**.

In [1]:
# Parámetros. papermill puede sobrescribirlos:  papermill 01_pipeline_raw_a_master.ipynb salida.ipynb -p RUTA_DB otra.db
# RUTA_OPERACIONAL: None usa data/external/retail.db (la crea el cuaderno 00), o la variable de entorno CURSO_DB_OPERACIONAL
RUTA_OPERACIONAL = None
# RUTA_DB: None usa data/processed/curso.db, o la variable de entorno CURSO_DB
RUTA_DB = None

### Preparación: dos bases de datos
Este cuaderno trabaja con **dos** bases: lee de la **operacional** (`retail.db`, el origen) y escribe en la **analítica** (`curso.db`, el destino).
La conexión principal es a `curso.db`; la operacional se adjuntará en el paso 1 solo para leerla.

In [2]:
import os, sqlite3, sys
from pathlib import Path

import pandas as pd

pd.set_option('display.width', 150, 'display.max_columns', 30)

# Carpeta del proyecto: la que contiene «sql» y «pyproject.toml».
RAIZ = Path.cwd().resolve()
while not ((RAIZ / 'sql').exists() and (RAIZ / 'pyproject.toml').exists()) and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
SQL = RAIZ / 'sql'
DB_OPERACIONAL = Path(RUTA_OPERACIONAL or os.environ.get('CURSO_DB_OPERACIONAL') or RAIZ / 'data' / 'external' / 'retail.db')
DB = Path(RUTA_DB or os.environ.get('CURSO_DB') or RAIZ / 'data' / 'processed' / 'curso.db')
DB.parent.mkdir(parents=True, exist_ok=True)

con = sqlite3.connect(DB)
con.execute('PRAGMA foreign_keys = ON')


def ejecutar_script(archivo):
    """Ejecuta un archivo .sql de la carpeta sql/ (puede tener varias sentencias)."""
    con.executescript((SQL / archivo).read_text(encoding='utf-8'))
    con.commit()


def escalar(consulta, parametros=None):
    fila = con.execute(consulta, parametros or {}).fetchone()
    return None if fila is None else fila[0]


print('Proyecto:', RAIZ)
print('Base operacional (origen):', DB_OPERACIONAL)
print('Base analítica (destino):', DB)

Proyecto: C:\bk\HC-Cursos\curso-ml-ia-desarrollo-produccion\04-Proyecto-Base
Base operacional (origen): C:\bk\HC-Cursos\curso-ml-ia-desarrollo-produccion\04-Proyecto-Base\data\external\retail.db
Base analítica (destino): C:\bk\HC-Cursos\curso-ml-ia-desarrollo-produccion\04-Proyecto-Base\data\processed\curso.db


## 1. RAW: ingerir el operacional sin limpiar nada
**Ingesta** es copiar los datos del sistema de origen a la capa RAW. Aquí hacemos una **carga completa**: cada ejecución borra RAW y lo vuelve a copiar entero.
(En sistemas grandes se usa la **carga incremental**: solo lo nuevo o lo modificado desde la última vez. La idea es la misma.)

Cómo se hace en este proyecto:
1. `sql/raw/01_crear_raw.sql` crea **una tabla RAW por cada tabla de origen**, con las mismas columnas, más `_carga_id`.
   Las tablas RAW **no tienen claves ni restricciones**: RAW no valida, solo guarda. Si el origen trajera una línea sin factura, RAW la aceptaría y MASTER la descartaría con su motivo.
2. `ATTACH DATABASE` conecta `retail.db` a la misma sesión con el alias `retail`. Así la copia es **una consulta SQL**: `INSERT INTO raw_x SELECT … FROM retail.x`.
   El cuaderno solo **lee** de `retail`: nunca escribe en el sistema de origen.
3. Cada tabla copiada deja una fila en **`raw_carga`**: qué tabla, cuántas filas, cuándo y la **huella sha256 de su contenido**.

**Linaje:** cada registro de RAW conserva su clave de origen (`linea_id`, `factura_id`…) y `_carga_id`, que apunta a la carga en la que llegó.
Con eso se puede responder siempre «¿de dónde salió esta fila?».

**Huella:** si mañana repites la ingesta y la huella de una tabla es la misma, el origen no ha cambiado; si es distinta, algo cambió
(aunque el número de filas coincida). Es la forma más barata de detectar cambios.

In [3]:
import hashlib
from datetime import datetime

TABLAS_OPERACIONALES = ['clientes', 'productos', 'facturas', 'lineas_factura']   # en orden de dependencias


def huella_tabla(tabla):
    """(filas, sha256) del contenido de una tabla, recorrida en su orden de inserción."""
    h, n = hashlib.sha256(), 0
    for fila in con.execute(f'SELECT * FROM {tabla} ORDER BY rowid'):
        h.update(repr(fila).encode('utf-8'))
        n += 1
    return n, h.hexdigest()

In [4]:
if not DB_OPERACIONAL.exists():
    raise FileNotFoundError(f'No existe {DB_OPERACIONAL}. Ejecuta antes el cuaderno 00 (o: python -m ia_mlops.operacional)')

ejecutar_script('raw/01_crear_raw.sql')
con.execute('ATTACH DATABASE ? AS retail', (str(DB_OPERACIONAL),))
ahora = datetime.now().isoformat(timespec='seconds')
for tabla in TABLAS_OPERACIONALES:
    filas, huella = huella_tabla(f'retail.{tabla}')
    carga = con.execute('INSERT INTO raw_carga (origen, tabla, filas, sha256, extraido_en) VALUES (?,?,?,?,?)',
                        ('retail', tabla, filas, huella, ahora)).lastrowid
    con.execute(f'INSERT INTO raw_{tabla} SELECT t.*, ? FROM retail.{tabla} AS t ORDER BY t.rowid', (carga,))
    print(f'raw_{tabla:15s} {filas:>10,} filas')
con.commit()
con.execute('DETACH DATABASE retail')
pd.read_sql('SELECT * FROM raw_carga', con)

raw_clientes               150 filas
raw_productos            3,080 filas
raw_facturas             1,430 filas


raw_lineas_factura      23,070 filas


,carga_id,origen,tabla,filas,sha256,extraido_en
0,1,retail,clientes,150,c890e667e675f961b9bb16e0ea2285394a4890df725ad6...,2026-11-11T10:27:16
1,2,retail,productos,3080,dba7cc6e947ff3b10d42f41843a98f6bfc1bc864f3a781...,2026-11-11T10:27:16
2,3,retail,facturas,1430,05192fce6cf74248219d62e9eb86c037987998692fa776...,2026-11-11T10:27:16
3,4,retail,lineas_factura,23070,d085b7896242aa6eec23742713df282269674e384b7994...,2026-11-11T10:27:16


In [5]:
# Linaje: linea_id es la clave de la línea en el operacional; _carga_id dice en qué carga llegó.
pd.read_sql('SELECT * FROM raw_lineas_factura LIMIT 8', con)

,linea_id,factura_id,codigo_producto,descripcion,cantidad,precio_unitario,registrada_en,_carga_id
0,1,C489530,72803A,ROSE SCENT CANDLE JEWELLED DRAWER,-1,4.25,2009-12-01 11:56:00,4
1,2,489547,22086,PAPER CHAIN KIT 50'S CHRISTMAS,18,2.95,2009-12-01 12:30:00,4
2,3,489547,21871,SAVE THE PLANET MUG,12,1.25,2009-12-01 12:30:00,4
3,4,489547,21875,KINGS CHOICE MUG,12,1.25,2009-12-01 12:30:00,4
4,5,489547,21587,COSY HOUR GIANT TUBE MATCHES,12,2.55,2009-12-01 12:30:00,4
5,6,489547,21593,SWALLOW CIGAR BOX MATCHES,24,1.25,2009-12-01 12:30:00,4
6,7,489547,21591,COSY HOUR CIGAR BOX MATCHES,24,1.25,2009-12-01 12:30:00,4
7,8,489547,21590,KINGS CHOICE CIGAR BOX MATCHES,24,1.25,2009-12-01 12:30:00,4


## 2. Mirar el dato crudo con ojo crítico
Antes de limpiar hay que **medir** qué está sucio (en la jerga, *perfilar* el dato). Nunca se limpia «a ojo»: cada hallazgo de esta sección
se convierte después en **una regla de MASTER** que se puede leer, discutir y cambiar.

La unidad de análisis es la **línea** (`raw_lineas_factura`) con los datos de su cabecera (`raw_facturas`), unidas con `LEFT JOIN`.
`LEFT JOIN` conserva todas las líneas aunque no encuentren su factura: así no escondemos ningún problema.

Qué buscamos y por qué importa:
- **Cancelaciones** (`C…`): son devoluciones, no ventas. Contarlas como ventas inflaría el gasto del cliente.
- **Sin cliente**: compras anónimas. No se pueden atribuir a nadie, así que no entran en las variables por cliente.
- **Cantidad o precio ≤ 0**: devoluciones, ajustes o regalos.
- **Códigos que no son productos**: `POST` (envío), `M` (manual), `BANK CHARGES`… No son artículos vendidos.
- **Duplicados exactos**: la misma línea registrada dos veces.

In [6]:
LINEAS = 'raw_lineas_factura l LEFT JOIN raw_facturas f ON f.factura_id = l.factura_id'
hallazgos = {
    'líneas': f"SELECT COUNT(*) FROM {LINEAS}",
    'facturas canceladas (empiezan por C)': f"SELECT COUNT(*) FROM {LINEAS} WHERE l.factura_id LIKE 'C%'",
    'sin cliente': f"SELECT COUNT(*) FROM {LINEAS} WHERE f.cliente_id IS NULL",
    'cantidad <= 0': f"SELECT COUNT(*) FROM {LINEAS} WHERE l.cantidad <= 0",
    'precio <= 0': f"SELECT COUNT(*) FROM {LINEAS} WHERE l.precio_unitario <= 0",
    'codigos que no empiezan por 5 digitos': f"SELECT COUNT(*) FROM {LINEAS} WHERE UPPER(TRIM(l.codigo_producto)) NOT GLOB '[0-9][0-9][0-9][0-9][0-9]*'",
    'descripcion nula o vacia': f"SELECT COUNT(*) FROM {LINEAS} WHERE l.descripcion IS NULL OR TRIM(l.descripcion) = ''",
}
total_lineas = escalar(hallazgos['líneas'])
pd.DataFrame({'lineas': {k: escalar(v) for k, v in hallazgos.items()}}).assign(
    porcentaje=lambda d: (100 * d['lineas'] / total_lineas).round(2))

,lineas,porcentaje
líneas,23070,100.00
facturas canceladas (empiezan por C),404,1.75
sin cliente,300,1.30
cantidad <= 0,408,1.77
precio <= 0,7,0.03
codigos que no empiezan por 5 digitos,93,0.40
descripcion nula o vacia,6,0.03


In [7]:
# Duplicados exactos: la misma línea registrada dos veces. Se conserva la primera y las demás se descartan con su motivo.
escalar(f"""
    SELECT COUNT(*) FROM (
      SELECT ROW_NUMBER() OVER (PARTITION BY l.factura_id, l.codigo_producto, l.descripcion, l.cantidad, l.registrada_en,
                                l.precio_unitario, IFNULL(f.cliente_id, -1), f.pais ORDER BY l.linea_id) AS rn
      FROM {LINEAS}) WHERE rn > 1""")

953

In [8]:
# Códigos que NO son productos: envío, comisiones, ajustes, regalos, pruebas...
pd.read_sql("""
    SELECT UPPER(TRIM(codigo_producto)) AS codigo, MIN(descripcion) AS descripcion, COUNT(*) AS lineas
    FROM raw_lineas_factura
    WHERE UPPER(TRIM(codigo_producto)) NOT GLOB '[0-9][0-9][0-9][0-9][0-9]*'
    GROUP BY 1 ORDER BY lineas DESC LIMIT 10""", con)

,codigo,descripcion,lineas
0,POST,POSTAGE,64
1,M,Manual,29


## 3. MASTER: depurar y modelar
`sql/master/01_crear_master.sql` crea las tablas (con claves y restricciones) y `02_poblar_master.sql` las llena desde RAW.
**Ábrelos y léelos: son SQL de verdad.** Las reglas, en orden, y por qué existen:

| # | Regla | Por qué |
|---|---|---|
| 1 | Recortar espacios (`TRIM`) y pasar códigos y descripciones a mayúsculas (`UPPER`) | `'85123a '` y `'85123A'` son el mismo producto |
| 2 | Descartar **duplicados exactos** (se conserva la primera) | Una línea repetida duplicaría la venta |
| 2 | Descartar filas con **datos obligatorios nulos** | Sin factura, producto, fecha, cantidad, precio o país no hay venta que analizar |
| 3 | Separar en **clientes, productos, facturas y líneas** | Modelo relacional: cada dato en su tabla, con claves |
| 4 | `es_cancelacion = 1` si la factura empieza por `C` | Las devoluciones no son compras |
| 5 | `es_producto = 1` si el código empieza por 5 dígitos | `POST` o `BANK CHARGES` no son artículos |
| 6 | `es_venta_valida = 1` si cantidad > 0, precio > 0, no es cancelación y es producto | Es la definición de «compra» que usarán FS y ML |

**Nada se borra en silencio:** lo que no pasa a MASTER queda en `master_descartes` con su **motivo**.
Las líneas que no son venta válida **sí** pasan a MASTER, pero marcadas con `es_venta_valida = 0`: así se pueden estudiar.

**Cómo se detecta un duplicado.** `ROW_NUMBER() OVER (PARTITION BY <todas las columnas> ORDER BY linea_id)` numera las filas idénticas
(1, 2, 3…) en el orden en que llegaron. La que tiene número 1 se queda; las demás se descartan. Es una **función de ventana**: calcula algo
para cada fila mirando a otras filas de su mismo grupo, sin agruparlas.

In [9]:
ejecutar_script('master/01_crear_master.sql')
ejecutar_script('master/02_poblar_master.sql')
for tabla in ('master_clientes', 'master_productos', 'master_facturas', 'master_lineas', 'master_descartes'):
    print(f'{tabla:18s} {escalar(f"SELECT COUNT(*) FROM {tabla}"):>10,} filas')

master_clientes           150 filas
master_productos        3,077 filas
master_facturas         1,430 filas
master_lineas          22,117 filas
master_descartes          953 filas


## 4. Reglas de calidad
Una regla de calidad es una **consulta que devuelve un número** y un criterio que dice si ese número es aceptable. Hay tres tipos:

| Tipo | Si falla… | Ejemplo |
|---|---|---|
| **Duro** | El pipeline **se detiene** (`ERROR`): seguir produciría resultados falsos | R01: las líneas de RAW deben ser exactamente las de MASTER más las descartadas |
| **Alerta** | Se registra y se avisa (`ALERTA`), pero se sigue | R06: facturas cuyas líneas tienen fechas distintas (MASTER toma la primera) |
| **Informativo** | Solo informa (`OK`) | R08: porcentaje de líneas sin cliente |

El resultado queda en la tabla `master_calidad`, con fecha: es la **evidencia** de que el dato se revisó. La misma lista de reglas está en
`ia_mlops/calidad.py`; las pruebas comprueban que el cuaderno y el paquete producen exactamente la misma tabla.

In [10]:
FECHA_MINIMA, FECHA_LIMITE = '2009-12-01', '2012-01-01'   # el límite es exclusivo
REGLAS = [  # (regla, descripción, consulta, tipo, umbral, cumple)
    ('R01_conciliacion', 'Líneas de RAW menos (líneas de MASTER + descartes). Debe ser 0.',
     "SELECT (SELECT COUNT(*) FROM raw_lineas_factura) - (SELECT COUNT(*) FROM master_lineas) - (SELECT COUNT(*) FROM master_descartes)", 'duro', '= 0', lambda v: v == 0),
    ('R02_facturas_sin_lineas', 'Facturas que no tienen ninguna línea.',
     "SELECT COUNT(*) FROM master_facturas f WHERE NOT EXISTS (SELECT 1 FROM master_lineas l WHERE l.factura_id = f.factura_id)", 'duro', '= 0', lambda v: v == 0),
    ('R03_integridad_referencial', 'Violaciones de claves foráneas.', None, 'duro', '= 0', lambda v: v == 0),
    ('R04_monto_coherente', 'Líneas donde monto_linea no es cantidad por precio.',
     "SELECT COUNT(*) FROM master_lineas WHERE ABS(monto_linea - ROUND(cantidad * precio_unitario, 2)) > 0.01", 'duro', '= 0', lambda v: v == 0),
    ('R05_fechas_en_rango', f'Facturas fuera del rango {FECHA_MINIMA} a {FECHA_LIMITE} (exclusivo).',
     f"SELECT COUNT(*) FROM master_facturas WHERE fecha_factura < '{FECHA_MINIMA}' OR fecha_factura >= '{FECHA_LIMITE}'", 'duro', '= 0', lambda v: v == 0),
    ('R06_facturas_con_varias_fechas', 'Facturas de RAW cuyas líneas tienen fechas distintas (MASTER toma la primera).',
     "SELECT COUNT(*) FROM (SELECT TRIM(factura_id) FROM raw_lineas_factura GROUP BY TRIM(factura_id) HAVING COUNT(DISTINCT registrada_en) > 1)", 'alerta', '= 0', lambda v: v == 0),
    ('R07_cancelaciones_con_cantidad_positiva', 'Líneas de facturas canceladas con cantidad positiva.',
     "SELECT COUNT(*) FROM master_lineas l JOIN master_facturas f ON f.factura_id = l.factura_id WHERE f.es_cancelacion = 1 AND l.cantidad > 0", 'alerta', '= 0', lambda v: v == 0),
    ('R08_pct_lineas_sin_cliente', 'Porcentaje de líneas cuya factura no trae Customer ID (no entran en FS).',
     "SELECT ROUND(100.0 * SUM(CASE WHEN f.cliente_id IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) FROM master_lineas l JOIN master_facturas f ON f.factura_id = l.factura_id", 'info', 'informativo', None),
    ('R09_pct_ventas_validas', 'Porcentaje de líneas que son ventas válidas.',
     "SELECT ROUND(100.0 * SUM(es_venta_valida) / COUNT(*), 2) FROM master_lineas", 'info', 'informativo', None),
    ('R10_duplicados_descartados', 'Filas descartadas por ser duplicados exactos.',
     "SELECT COUNT(*) FROM master_descartes WHERE motivo = 'duplicado_exacto'", 'info', 'informativo', None),
    ('R11_lineas_precio_no_positivo', 'Líneas con precio menor o igual a cero.',
     "SELECT COUNT(*) FROM master_lineas WHERE precio_unitario <= 0", 'info', 'informativo', None),
    ('R12_codigos_que_no_son_producto', 'Códigos de StockCode que no son productos (envío, comisiones, ajustes...).',
     "SELECT COUNT(*) FROM master_productos WHERE es_producto = 0", 'info', 'informativo', None),
    ('R13_clientes_distintos', 'Clientes distintos en MASTER.', "SELECT COUNT(*) FROM master_clientes", 'info', 'informativo', None),
]

ahora = datetime.now().isoformat(timespec='seconds')
con.execute('DELETE FROM master_calidad')
errores = []
for regla, descripcion, consulta, tipo, umbral, cumple in REGLAS:
    valor = len(con.execute('PRAGMA foreign_key_check').fetchall()) if consulta is None else con.execute(consulta).fetchone()[0]
    estado = 'OK' if tipo == 'info' else ('OK' if cumple(valor) else ('ERROR' if tipo == 'duro' else 'ALERTA'))
    if estado == 'ERROR':
        errores.append(f'{regla}: {descripcion} -> {valor}')
    con.execute('INSERT INTO master_calidad (regla, descripcion, valor, umbral, estado, evaluado_en) VALUES (?,?,?,?,?,?)',
                (regla, descripcion, valor, umbral, estado, ahora))
con.commit()
assert not errores, 'Fallaron reglas de calidad duras:\n  ' + '\n  '.join(errores)
pd.read_sql('SELECT regla, valor, umbral, estado FROM master_calidad ORDER BY regla', con)

,regla,valor,umbral,estado
0,R01_conciliacion,0.00,= 0,OK
1,R02_facturas_sin_lineas,0.00,= 0,OK
2,R03_integridad_referencial,0.00,= 0,OK
3,R04_monto_coherente,0.00,= 0,OK
4,R05_fechas_en_rango,0.00,= 0,OK
5,R06_facturas_con_varias_fechas,1.00,= 0,ALERTA
6,R07_cancelaciones_con_cantidad_positiva,0.00,= 0,OK
7,R08_pct_lineas_sin_cliente,1.36,informativo,OK
8,R09_pct_ventas_validas,97.82,informativo,OK
9,R10_duplicados_descartados,953.00,informativo,OK


### Nada desaparece sin dejar rastro
Lo que no pasó a MASTER está en `master_descartes`, con su motivo. La regla R01 lo comprueba con una **conciliación**:
`líneas de RAW = líneas de MASTER + descartes`. Si alguna vez no cuadra, el pipeline se detiene: una fila se ha perdido por el camino.

In [11]:
pd.read_sql("""
    SELECT 'RAW (líneas)' AS capa, COUNT(*) AS filas FROM raw_lineas_factura
    UNION ALL SELECT 'MASTER (líneas)', COUNT(*) FROM master_lineas
    UNION ALL SELECT 'descartes: ' || motivo, COUNT(*) FROM master_descartes GROUP BY motivo""", con)

,capa,filas
0,RAW (líneas),23070
1,MASTER (líneas),22117
2,descartes: duplicado_exacto,953


In [12]:
pd.read_sql("""
    SELECT f.factura_id, f.fecha_factura, f.cliente_id, l.codigo_producto, p.descripcion, l.cantidad, l.precio_unitario,
           l.monto_linea, l.es_venta_valida
    FROM master_lineas l
    JOIN master_facturas f ON f.factura_id = l.factura_id
    JOIN master_productos p ON p.codigo_producto = l.codigo_producto
    ORDER BY l.linea_id LIMIT 8""", con)

,factura_id,fecha_factura,cliente_id,codigo_producto,descripcion,cantidad,precio_unitario,monto_linea,es_venta_valida
0,C489530,2009-12-01 11:56:00,14748,72803A,ROSE SCENT CANDLE JEWELLED DRAWER,-1,4.25,-4.25,0
1,489547,2009-12-01 12:30:00,13599,22086,PAPER CHAIN KIT 50'S CHRISTMAS,18,2.95,53.10,1
2,489547,2009-12-01 12:30:00,13599,21871,SAVE THE PLANET MUG,12,1.25,15.00,1
3,489547,2009-12-01 12:30:00,13599,21875,KINGS CHOICE MUG,12,1.25,15.00,1
4,489547,2009-12-01 12:30:00,13599,21587,COSY HOUR GIANT TUBE MATCHES,12,2.55,30.60,1
5,489547,2009-12-01 12:30:00,13599,21593,SWALLOW CIGAR BOX MATCHES,24,1.25,30.00,1
6,489547,2009-12-01 12:30:00,13599,21591,COSY HOUR CIGAR BOX MATCHES,24,1.25,30.00,1
7,489547,2009-12-01 12:30:00,13599,21590,KINGS CHOICE CIGAR BOX MATCHES,24,1.25,30.00,1


## Resumen
- **RAW** es la copia fiel del operacional: una tabla por tabla de origen, con linaje (`_carga_id`) y huella de cada carga en `raw_carga`.
- **MASTER** es el dato depurado: duplicados y nulos descartados con motivo, cada línea marcada como venta válida o no, y un modelo relacional con claves.
- Las **reglas de calidad** demuestran que el resultado es coherente; si una regla dura falla, el pipeline se detiene.

**Siguiente:** el cuaderno **02** calcula, a partir de MASTER, las variables de cada cliente en cada fecha de corte (capa FS).

In [13]:
con.close()
print('Listo: RAW y MASTER construidos en', DB)

Listo: RAW y MASTER construidos en C:\bk\HC-Cursos\curso-ml-ia-desarrollo-produccion\04-Proyecto-Base\data\processed\curso.db


## Para practicar (tarea de S1)
1. Añade una **regla de calidad** nueva a la lista `REGLAS` (por ejemplo, ventas con un monto desproporcionado) y confirma que aparece en `master_calidad`.
2. Cambia una regla de limpieza en `sql/master/02_poblar_master.sql` (por ejemplo, qué cuenta como producto) y vuelve a ejecutar. ¿Cambió la conciliación?
3. Explica en dos frases por qué RAW no se corrige, aunque el operacional tenga errores.
4. Vuelve a ejecutar la ingesta sin cambiar el operacional. ¿Cambian las huellas de `raw_carga`? ¿Y si borras una línea en `retail.db`?